## 1. Mount Google Drive
Persist matched-volume experiment artifacts to Drive for later local analysis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/afriqa_outputs_matchedqa

## 2. Install Dependencies

In [ ]:
import os

if not os.path.exists('afriqa-entity-aware-qa'):
    !git clone https://github.com/OmondiKevin/afriqa-entity-aware-qa.git

%cd afriqa-entity-aware-qa
!git pull
!pip install -e .
!pip install sentence-transformers sentencepiece protobuf peft

## 3. Configure Environment and Outputs
Use a Drive-backed `outputs` symlink so all logs/metrics/predictions are persisted.

In [ ]:
import os
import torch
from transformers import set_seed

set_seed(42)

# Keep outputs persisted on Drive for easy pull-back to local machine.
if os.path.exists('outputs'):
    !rm -rf outputs
!ln -s /content/drive/MyDrive/afriqa_outputs_matchedqa outputs

# GPU runtime optimization knobs.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1))
else:
    print('Warning: CUDA not detected. Colab runtime is likely not on GPU.')

## 4. Download Dataset
Download AfriQA and MasakhaNER resources used by the repository pipeline.

In [ ]:
!python scripts/00_download_and_subset.py --config configs/default.yaml

## 5. Prepare QA Seq2Seq Data
Build QA-only seq2seq JSONL files used by the matched-volume ablation.

In [ ]:
!python scripts/01_prepare_qa_data.py --config configs/default.yaml

## 6. Create Matched Ablation Configs
Create Colab-local configs with explicit `matchedqa_*` output paths and QA upsampling control.

In [ ]:
import copy
import yaml
import torch
from pathlib import Path

RUN_BYT5 = True

base_cfg_path = Path('configs/default.yaml')
base_cfg = yaml.safe_load(base_cfg_path.read_text(encoding='utf-8'))

# Choose the fastest safe precision for the current GPU runtime.
use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
use_fp16 = bool(torch.cuda.is_available() and not use_bf16)

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if vram_gb >= 39:
        tuned_batch_size = 64
    elif vram_gb >= 23:
        tuned_batch_size = 32
    else:
        tuned_batch_size = 16
else:
    tuned_batch_size = 8

cfg_mt5 = copy.deepcopy(base_cfg)
cfg_mt5.setdefault('model', {})['base'] = 'google/mt5-base'
cfg_mt5.setdefault('ablation', {})['matched_qa_upsample_factor'] = int(cfg_mt5.get('multitask', {}).get('qa_upsample_factor', 20))
cfg_mt5.setdefault('run', {})['matchedqa_output_dir'] = 'outputs/checkpoints/matchedqa_mt5'
cfg_mt5['run']['matchedqa_pred_path'] = 'outputs/predictions/matchedqa_mt5_test.jsonl'
cfg_mt5['run']['matchedqa_log_path'] = 'outputs/logs/02_train_matchedqa_mt5.log'

cfg_mt5.setdefault('train', {})['early_stopping_patience'] = int(cfg_mt5.get('train', {}).get('early_stopping_patience', 3))
cfg_mt5['train']['batch_size'] = tuned_batch_size
cfg_mt5['train']['grad_accum'] = 1
cfg_mt5['train']['bf16'] = use_bf16
cfg_mt5['train']['fp16'] = use_fp16

Path('configs/colab_matchedqa_mt5.yaml').write_text(yaml.safe_dump(cfg_mt5, sort_keys=False, allow_unicode=False), encoding='utf-8')

if RUN_BYT5:
    cfg_byt5 = copy.deepcopy(base_cfg)
    cfg_byt5.setdefault('model', {})['base'] = 'google/byt5-base'
    cfg_byt5.setdefault('ablation', {})['matched_qa_upsample_factor'] = int(cfg_byt5.get('multitask', {}).get('qa_upsample_factor', 20))
    cfg_byt5.setdefault('run', {})['matchedqa_output_dir'] = 'outputs/checkpoints/matchedqa_byt5'
    cfg_byt5['run']['matchedqa_pred_path'] = 'outputs/predictions/matchedqa_byt5_test.jsonl'
    cfg_byt5['run']['matchedqa_log_path'] = 'outputs/logs/02_train_matchedqa_byt5.log'

    cfg_byt5.setdefault('train', {})['early_stopping_patience'] = int(cfg_byt5.get('train', {}).get('early_stopping_patience', 3))
    cfg_byt5['train']['batch_size'] = tuned_batch_size
    cfg_byt5['train']['grad_accum'] = 1
    cfg_byt5['train']['bf16'] = use_bf16
    cfg_byt5['train']['fp16'] = use_fp16

    Path('configs/colab_matchedqa_byt5.yaml').write_text(yaml.safe_dump(cfg_byt5, sort_keys=False, allow_unicode=False), encoding='utf-8')

print('Wrote configs:')
print(' - configs/colab_matchedqa_mt5.yaml')
if RUN_BYT5:
    print(' - configs/colab_matchedqa_byt5.yaml')
print('Precision -> bf16:', use_bf16, 'fp16:', use_fp16)
print('Tuned batch_size:', tuned_batch_size)

!cat configs/colab_matchedqa_mt5.yaml | grep -A 3 "model:"
!cat configs/colab_matchedqa_mt5.yaml | grep -A 10 "ablation:"
!cat configs/colab_matchedqa_mt5.yaml | grep -A 14 "train:"
!cat configs/colab_matchedqa_mt5.yaml | grep -A 12 "run:"

## 7. Train Matched-Volume QA-Only (`mT5-base`)
Run baseline trainer in `--matched_volume` mode (upsampled QA only, no NER supervision).

In [ ]:
!python scripts/02_train_baseline_qa.py --config configs/colab_matchedqa_mt5.yaml --matched_volume

## 8. Evaluate Matched-Volume QA-Only (`mT5-base`)
Write metrics using the prediction stem and a dedicated eval log.

In [ ]:
!python scripts/04_eval_predictions.py --config configs/colab_matchedqa_mt5.yaml --pred_path outputs/predictions/matchedqa_mt5_test.jsonl --log_path outputs/logs/04_eval_predictions_matchedqa_mt5.log

## 9. Optional: Train Matched-Volume QA-Only (`ByT5-base`)

In [ ]:
import os

if os.path.exists('configs/colab_matchedqa_byt5.yaml'):
    !python scripts/02_train_baseline_qa.py --config configs/colab_matchedqa_byt5.yaml --matched_volume
else:
    print('Skipping ByT5 matched training (config not found).')

## 10. Optional: Evaluate Matched-Volume QA-Only (`ByT5-base`)

In [ ]:
import os

if os.path.exists('configs/colab_matchedqa_byt5.yaml'):
    !python scripts/04_eval_predictions.py --config configs/colab_matchedqa_byt5.yaml --pred_path outputs/predictions/matchedqa_byt5_test.jsonl --log_path outputs/logs/04_eval_predictions_matchedqa_byt5.log
else:
    print('Skipping ByT5 matched evaluation (config not found).')

## 11. Verify Saved Artifacts
Confirm checkpoints, predictions, metrics, and logs were written to Drive-backed outputs.

In [ ]:
!ls -lh outputs/checkpoints
!ls -lh outputs/predictions
!ls -lh outputs/metrics
!ls -lh outputs/logs | sed -n '1,200p'

## 12. Display Final Metrics Summary
Show overall scores for matched ablations and any existing baseline/multitask metrics present.

In [ ]:
import json
import os

metrics_map = {
    'matchedqa_mt5': 'outputs/metrics/matchedqa_mt5_test_metrics.json',
    'matchedqa_byt5': 'outputs/metrics/matchedqa_byt5_test_metrics.json',
    'baseline_mt5': 'outputs/metrics/baseline_mt5_test_metrics.json',
    'multitask_mt5_qa_only': 'outputs/metrics/multitask_mt5_test_qa_only_metrics.json',
    'multitask_byt5_qa_only': 'outputs/metrics/multitask_byt5_test_qa_only_metrics.json',
}

for name, path in metrics_map.items():
    print(f'\n=== {name} ===')
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(json.dumps(data.get('overall', {}), indent=2, ensure_ascii=False))
    else:
        print(f'Missing metrics file: {path}')